# Chest X-ray Classification: Hybrid Model & Grad-CAM

This notebook implements a Hybrid VGG16 + DenseNet121 model for multi-label classification of Chest X-rays.
It automatically scans the data directory for images, filters the dataset, and applies a 70/10/20 train/val/test split.

In [8]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import VGG16, DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
DATA_ROOT = "D:/VitalScanAI/backend/data/dataset"
CSV_PATH = "Data_Entry_2017.csv"
BBOX_PATH = "BBox_List_2017.csv"
EXPORT_DIR = "./models_export/xray"
os.makedirs(EXPORT_DIR, exist_ok=True)

## 1. Data Discovery and Loading
We scan the `DATA_ROOT` recursively to find all images, creating a mapping from filename to absolute path.

In [9]:
# 1. Scan for all images
image_paths = glob.glob(os.path.join(DATA_ROOT, '**', '*.png'), recursive=True)
print(f"Found {len(image_paths)} images in {DATA_ROOT}")

if len(image_paths) == 0:
    raise ValueError("No images found! Please check download path.")

# Map filename -> full path
path_map = {os.path.basename(p): p for p in image_paths}

# 2. Load Metadata Tables
try:
    label_df = pd.read_csv(CSV_PATH)
    bbox_df = pd.read_csv(BBOX_PATH)
    print(f"Entries loaded: {len(label_df)}")
    print(f"BBox entries loaded: {len(bbox_df)}")
except FileNotFoundError as e:
    print(f"Error loading CSVs: {e}")
    label_df = pd.DataFrame(columns=['Image Index', 'Finding Labels'])

# 3. Filter DataFrame to maintain only existing images
# This step effectively intersects the metadata with the files we actually have on disk
label_df['path'] = label_df['Image Index'].map(path_map)
# Drop rows where path is NaN (image not found)
label_df = label_df.dropna(subset=['path'])

print(f"Matched Dataset Size: {len(label_df)}")

# 4. Process Labels
label_df['labels'] = label_df['Finding Labels'].apply(lambda x: x.split('|'))
mlb = MultiLabelBinarizer()
mlb.fit(label_df['labels'])
classes = mlb.classes_
print(f"Classes ({len(classes)}): {classes}")

# 5. Split Logic: 70% Train, 10% Val, 20% Test
if len(label_df) > 0:
    # Split 1: Split 20% out for Test directly
    train_val_df, test_df = train_test_split(label_df, test_size=0.20, random_state=42)
    
    # Split 2: From the remaining 80%, we want 10% of TOTAL to be validation.
    # 10% is 1/8th of 80%. So test_size = 0.125
    train_df, val_df = train_test_split(train_val_df, test_size=0.125, random_state=42)
    
    print("\n--- Split Statistics ---")
    print(f"Total: {len(label_df)}")
    print(f"Training:   {len(train_df)} ({len(train_df)/len(label_df):.1%})")
    print(f"Validation: {len(val_df)} ({len(val_df)/len(label_df):.1%})")
    print(f"Testing:    {len(test_df)} ({len(test_df)/len(label_df):.1%})")
else:
    print("No matching data available after filtering.")
    train_df, val_df, test_df = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

Found 9529 images in D:/VitalScanAI/backend/data/dataset
Entries loaded: 112120
BBox entries loaded: 984
Matched Dataset Size: 9529
Classes (15): ['Atelectasis' 'Cardiomegaly' 'Consolidation' 'Edema' 'Effusion'
 'Emphysema' 'Fibrosis' 'Hernia' 'Infiltration' 'Mass' 'No Finding'
 'Nodule' 'Pleural_Thickening' 'Pneumonia' 'Pneumothorax']

--- Split Statistics ---
Total: 9529
Training:   6670 (70.0%)
Validation: 953 (10.0%)
Testing:    1906 (20.0%)


In [10]:
# Generators
# Note: x_col='path' allows us to point to files in different subdirectories!
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

print("Creating Training Generator...")
if len(train_df) > 0:
    train_gen = train_datagen.flow_from_dataframe(
        dataframe=train_df,
        x_col='path', # Use absolute path column
        y_col='labels',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=list(classes) 
    )

    print("Creating Validation Generator...")
    val_gen = val_datagen.flow_from_dataframe(
        dataframe=val_df,
        x_col='path',
        y_col='labels',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=list(classes),
        shuffle=False
    )

    print("Creating Test Generator...")
    test_gen = val_datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='path',
        y_col='labels',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=list(classes),
        shuffle=False
    )
else:
    print("Skipping generator creation: No training data.")

Creating Training Generator...
Found 6670 validated image filenames belonging to 15 classes.
Creating Validation Generator...
Found 953 validated image filenames belonging to 15 classes.
Creating Test Generator...
Found 1906 validated image filenames belonging to 15 classes.


## 2. Hybrid Model Architectures

In [11]:
def build_hybrid_model(input_shape, num_classes):
    input_tensor = Input(shape=input_shape)
    
    # --- Branch 1: VGG16 ---
    vgg = VGG16(weights='imagenet', include_top=False, input_tensor=input_tensor)
    for layer in vgg.layers:
        layer.trainable = False
        layer._name = f"vgg_{layer.name}"
        
    vgg_out = vgg.output
    vgg_pool = GlobalAveragePooling2D(name='vgg_gap')(vgg_out)
    
    # --- Branch 2: DenseNet121 ---
    densenet = DenseNet121(weights='imagenet', include_top=False, input_tensor=input_tensor)
    for layer in densenet.layers:
        layer.trainable = False
        layer._name = f"dense_{layer.name}"

    densenet_out = densenet.output
    densenet_pool = GlobalAveragePooling2D(name='densenet_gap')(densenet_out)
    
    # --- Fusion ---
    concat = Concatenate(name='fusion_concat')([vgg_pool, densenet_pool])
    
    x = Dense(512, activation='relu')(concat)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    
    output = Dense(num_classes, activation='sigmoid', name='visual_output')(x)
    
    model = Model(inputs=input_tensor, outputs=output, name="Hybrid_VGG_DenseNet")
    return model

if len(classes) > 0:
    hybrid_model = build_hybrid_model(IMG_SIZE + (3,), len(classes))
    hybrid_model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True, name='auc')]
    )
    hybrid_model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


Model: "Hybrid_VGG_DenseNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 22,674,319 (86.50 MB)

 Trainable params: 922,127 (3.52 MB)

 Non-trainable params: 21,752,192 (82.98 MB)

In [12]:
# Callbacks
checkpoint = ModelCheckpoint(
    os.path.join(EXPORT_DIR, 'best_hybrid_model.h5'), 
    monitor='val_auc', 
    save_best_only=True, 
    mode='max', 
    verbose=1
)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

# Training
print("Starting Training...")
if 'train_gen' in locals():
    history = hybrid_model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=[checkpoint, early_stop, reduce_lr]
    )
else:
    print("Skipping training: No data available.")

Starting Training...
Epoch 1/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.3634 - auc: 0.5175 - loss: 0.3615
Epoch 1: val_auc improved from None to 0.57739, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 1: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1200s 6s/step - accuracy: 0.4688 - auc: 0.5108 - loss: 0.2758 - val_accuracy: 0.5456 - val_auc: 0.5774 - val_loss: 0.2074 - learning_rate: 1.0000e-04
Epoch 2/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.5387 - auc: 0.5235 - loss: 0.2272
Epoch 2: val_auc improved from 0.57739 to 0.59420, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 2: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1273s 6s/step - accuracy: 0.5417 - auc: 0.5346 - loss: 0.2253 - val_accuracy: 0.5456 - val_auc: 0.5942 - val_loss: 0.2033 - learning_rate: 1.0000e-04
Epoch 3/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.5530 - auc: 0.5596 - loss: 0.2177
Epoch 3: val_auc improved from 0.59420 to 0.60829, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 3: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1174s 6s/step - accuracy: 0.5523 - auc: 0.5760 - loss: 0.2163 - val_accuracy: 0.5435 - val_auc: 0.6083 - val_loss: 0.2006 - learning_rate: 1.0000e-04
Epoch 4/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.5382 - auc: 0.5896 - loss: 0.2167
Epoch 4: val_auc improved from 0.60829 to 0.62008, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 4: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1169s 6s/step - accuracy: 0.5516 - auc: 0.5872 - loss: 0.2118 - val_accuracy: 0.5425 - val_auc: 0.6201 - val_loss: 0.1995 - learning_rate: 1.0000e-04
Epoch 5/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5614 - auc: 0.6092 - loss: 0.2068
Epoch 5: val_auc improved from 0.62008 to 0.63009, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 5: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1051s 5s/step - accuracy: 0.5505 - auc: 0.6077 - loss: 0.2094 - val_accuracy: 0.5446 - val_auc: 0.6301 - val_loss: 0.1995 - learning_rate: 1.0000e-04
Epoch 6/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5576 - auc: 0.6143 - loss: 0.2057
Epoch 6: val_auc improved from 0.63009 to 0.63341, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 6: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1025s 5s/step - accuracy: 0.5576 - auc: 0.6155 - loss: 0.2061 - val_accuracy: 0.5435 - val_auc: 0.6334 - val_loss: 0.1979 - learning_rate: 1.0000e-04
Epoch 7/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5595 - auc: 0.6249 - loss: 0.2055
Epoch 7: val_auc improved from 0.63341 to 0.64343, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 7: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1024s 5s/step - accuracy: 0.5574 - auc: 0.6258 - loss: 0.2055 - val_accuracy: 0.5425 - val_auc: 0.6434 - val_loss: 0.1982 - learning_rate: 1.0000e-04
Epoch 8/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5668 - auc: 0.6408 - loss: 0.2005
Epoch 8: val_auc improved from 0.64343 to 0.65146, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 8: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1036s 5s/step - accuracy: 0.5541 - auc: 0.6359 - loss: 0.2041 - val_accuracy: 0.5393 - val_auc: 0.6515 - val_loss: 0.1960 - learning_rate: 1.0000e-04
Epoch 9/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5536 - auc: 0.6339 - loss: 0.2000
Epoch 9: val_auc improved from 0.65146 to 0.65744, saving model to ./models_export/xray\best_hybrid_model.h5



Epoch 9: finished saving model to ./models_export/xray\best_hybrid_model.h5
209/209 ━━━━━━━━━━━━━━━━━━━━ 1024s 5s/step - accuracy: 0.5543 - auc: 0.6438 - loss: 0.2031 - val_accuracy: 0.5456 - val_auc: 0.6574 - val_loss: 0.2003 - learning_rate: 1.0000e-04
Epoch 10/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5420 - auc: 0.6163 - loss: 0.2102
Epoch 10: val_auc did not improve from 0.65744
209/209 ━━━━━━━━━━━━━━━━━━━━ 1026s 5s/step - accuracy: 0.5591 - auc: 0.6339 - loss: 0.2035 - val_accuracy: 0.5435 - val_auc: 0.6569 - val_loss: 0.1985 - learning_rate: 1.0000e-04


## 3. Grad-CAM Visualization

In [13]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, cam_path="cam.jpg", alpha=0.4):
    img = tf.keras.preprocessing.image.load_img(img_path)
    img = tf.keras.preprocessing.image.img_to_array(img)
    heatmap = np.uint8(255 * heatmap)
    jet = plt.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)
    superimposed_img = jet_heatmap * alpha + img
    superimposed_img = tf.keras.preprocessing.image.array_to_img(superimposed_img)
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img.astype('uint8') / 255)
    plt.title("Original")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(superimposed_img)
    plt.title("Grad-CAM")
    plt.axis("off")
    plt.show()

if 'test_df' in locals() and len(test_df) > 0:
    sample_row = test_df.iloc[0]
    # Since we have the full path map now, we can use the 'path' column directly
    sample_img_path = sample_row['path']
    img = tf.keras.preprocessing.image.load_img(sample_img_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    print("Visualizing Attention...")
    try:
        heatmap = make_gradcam_heatmap(img_array, hybrid_model, 'vgg_block5_conv3')
        display_gradcam(sample_img_path, heatmap)
    except Exception as e:
        print(f"Could not visualize: {e}")

Visualizing Attention...
Could not visualize: No such layer: vgg_block5_conv3. Existing layers are: ['input_layer', 'zero_padding2d', 'conv1_conv', 'conv1_bn', 'conv1_relu', 'zero_padding2d_1', 'pool1', 'conv2_block1_0_bn', 'conv2_block1_0_relu', 'conv2_block1_1_conv', 'conv2_block1_1_bn', 'conv2_block1_1_relu', 'conv2_block1_2_conv', 'conv2_block1_concat', 'conv2_block2_0_bn', 'conv2_block2_0_relu', 'conv2_block2_1_conv', 'conv2_block2_1_bn', 'conv2_block2_1_relu', 'conv2_block2_2_conv', 'conv2_block2_concat', 'conv2_block3_0_bn', 'conv2_block3_0_relu', 'conv2_block3_1_conv', 'conv2_block3_1_bn', 'conv2_block3_1_relu', 'conv2_block3_2_conv', 'conv2_block3_concat', 'conv2_block4_0_bn', 'conv2_block4_0_relu', 'conv2_block4_1_conv', 'conv2_block4_1_bn', 'conv2_block4_1_relu', 'conv2_block4_2_conv', 'conv2_block4_concat', 'conv2_block5_0_bn', 'conv2_block5_0_relu', 'conv2_block5_1_conv', 'conv2_block5_1_bn', 'conv2_block5_1_relu', 'conv2_block5_2_conv', 'conv2_block5_concat', 'conv2_block

In [14]:
# Export Model
if 'hybrid_model' in locals():
    export_path = os.path.join(EXPORT_DIR, "hybrid_chest_xray_model.keras")
    hybrid_model.save(export_path)
    print(f"Model saved to {export_path}")

Model saved to ./models_export/xray\hybrid_chest_xray_model.keras
